In [5]:
%run funcoes_rotina_final.py

=== INICIANDO EXPERIMENTO TOPICGPT (MODO COMPLETO) ===
Modelo selecionado: UnB-Qwen/qwen3-8-27b
Ambiente configurado! topicgpt_python apontará para: https://ia-router.unb.br/ (patch UnB ativo)
Testando conexão e modo de resposta no proxy da UnB (UnB-Qwen/qwen3-8-27b)...
Conexão validada com sucesso! Estratégia ativa: 'extra_body' (desabilitar thinking via chat_template_kwargs) | Resposta: 'OK'

DATASET 1: CONGRESSIONAL BILLS
[DOWNLOAD] Reutilizando datasets/topicgpt_oficial/bills_train.metadata.jsonl (76.5 MB).
[DOWNLOAD] Reutilizando datasets/topicgpt_oficial/bills_test.metadata.jsonl (54.1 MB).
[BILLS] bills_train.metadata.jsonl: 28882 documentos válidos | 21 labels.
[BILLS] bills_test.metadata.jsonl: 13848 documentos válidos | 19 labels.
[BILLS] Registros antes da união | Train: 28882 | Test: 13848
[BILLS] Base total após a união: 41930 documentos únicos
[BILLS] Duplicidades removidas: 800
[BILLS] Labels distintos: 21
[BILLS] Preparação concluída!
[BILLS] Geração: 1000 documentos | 

KeyboardInterrupt: 

## Somente Etapa de Geração para as duas bases

In [ ]:
from funcoes_rotina_final import (
    ConfiguradorOllama,
    PipelineMestrado
)

ConfiguradorOllama.setup_ambiente_topicgpt(
    host="127.0.0.1",
    port="11434"
)

pipeline = PipelineMestrado(model_name="gemma2:9b")

pipeline.atribuir_topicos(
    data_file="datasets/experimento/bills_ass.jsonl",
    prompt_file="prompts/assignment.txt",
    out_file="resultados/b_atribuida.json",
    topic_file="resultados/b_top_refinados.json"
)

## Somente Base Wiki

In [1]:
from pathlib import Path

arquivos_wiki_antigos = [
    "datasets/experimento/wiki_gen.jsonl",
    "datasets/experimento/wiki_ass.jsonl",
    "resultados/w_geracao.json",
    "resultados/w_topicos.json",
    "resultados/w_refinamento.json",
    "resultados/w_top_refinados.json",
    "resultados/w_top_refinados_limpo.txt",
    "resultados/w_map.json",
    "resultados/w_atribuida.json",
]

for arquivo in arquivos_wiki_antigos:
    caminho = Path(arquivo)

    if caminho.exists():
        caminho.unlink()
        print(f"Excluído: {arquivo}")
    else:
        print(f"Não existia: {arquivo}")

Não existia: datasets/experimento/wiki_gen.jsonl
Não existia: datasets/experimento/wiki_ass.jsonl
Não existia: resultados/w_geracao.json
Não existia: resultados/w_topicos.json
Não existia: resultados/w_refinamento.json
Não existia: resultados/w_top_refinados.json
Não existia: resultados/w_top_refinados_limpo.txt
Não existia: resultados/w_map.json
Não existia: resultados/w_atribuida.json


In [12]:
import importlib
import funcoes_rotina_final

importlib.invalidate_caches()
importlib.reload(funcoes_rotina_final)

from funcoes_rotina_final import (
    ConfiguradorOllama,
    PreparadorDatasets,
    PipelineMestrado,
    AvaliadorMetricas,
    registrar_log,
    MODO_TESTE,
)

print("Arquivo carregado:", funcoes_rotina_final.__file__)
print("MODO_TESTE:", MODO_TESTE)

Arquivo carregado: /home/tiagosampaio/experimento_mestrado/funcoes_rotina_final.py
MODO_TESTE: False


In [3]:
file_wiki_gen, file_wiki_ass, n_gen_w, n_ass_w = (
    PreparadorDatasets.preparar_wiki()
)

print("Arquivo de geração:", file_wiki_gen)
print("Arquivo de atribuição:", file_wiki_ass)
print("Quantidade para geração:", n_gen_w)
print("Quantidade para atribuição:", n_ass_w)

[DOWNLOAD] Reutilizando datasets/topicgpt_oficial/wiki_train.metadata.jsonl (318.8 MB).
[DOWNLOAD] Reutilizando datasets/topicgpt_oficial/wiki_test.metadata.jsonl (207.9 MB).
[WIKI] wiki_train.metadata.jsonl: 14235 documentos válidos | 45 labels.
[WIKI] wiki_test.metadata.jsonl: 7987 documentos válidos | 37 labels.
[WIKI] Registros antes da união | Train: 14235 | Test: 7987
[WIKI] Base total após a união: 22222 documentos únicos
[WIKI] Duplicidades removidas: 0
[WIKI] Labels distintos: 45
[WIKI] Preparação concluída!
[WIKI] Geração: 1100 documentos | 44 labels
[WIKI] Atribuição: 8024 documentos | 45 labels
[WIKI] Sobreposição entre as amostras: 0
[WIKI] Arquivo de geração: datasets/experimento/wiki_gen.jsonl
[WIKI] Arquivo de atribuição: datasets/experimento/wiki_ass.jsonl
Arquivo de geração: datasets/experimento/wiki_gen.jsonl
Arquivo de atribuição: datasets/experimento/wiki_ass.jsonl
Quantidade para geração: 1100
Quantidade para atribuição: 8024


In [4]:
import pandas as pd

df_wiki_gen = pd.read_json(
    file_wiki_gen,
    lines=True,
    engine="ujson",
)

df_wiki_ass = pd.read_json(
    file_wiki_ass,
    lines=True,
    engine="ujson",
)

print("Geração:", df_wiki_gen.shape)
print("Atribuição:", df_wiki_ass.shape)

print(
    "Labels distintos na geração:",
    df_wiki_gen["label_name"].nunique(dropna=True),
)

print(
    "Labels distintos na atribuição:",
    df_wiki_ass["label_name"].nunique(dropna=True),
)

print("\nDistribuição dos labels na atribuição:")

display(
    df_wiki_ass["label_name"]
    .value_counts(dropna=False)
    .rename_axis("label_name")
    .reset_index(name="quantidade")
)

Geração: (1100, 2)
Atribuição: (8024, 2)
Labels distintos na geração: 44
Labels distintos na atribuição: 45

Distribuição dos labels na atribuição:


,label_name,quantidade
0,Television,768
1,Transport,555
2,Songs,468
3,Biology and medicine,433
4,Warships and naval units,393
5,Earth science,347
6,Video games,336
7,World history,333
8,Albums,298
9,"Battles, exercises, and conflicts",276


In [5]:
import os

os.makedirs(
    "resultados",
    exist_ok=True,
)

ConfiguradorOllama.setup_ambiente_topicgpt(
    host="127.0.0.1",
    port="11434",
)

pipeline = PipelineMestrado(
    model_name="gemma2:9b",
)

print("Pipeline configurado.")

Ambiente configurado! topicgpt_python apontará para: http://127.0.0.1:11434/v1
Pipeline configurado.


In [5]:
qtd_topicos_wiki = pipeline.gerar_topicos(
    data_file=file_wiki_gen,
    prompt_file="prompts/generation_1.txt",
    seed_file="prompts/seed_1.md",
    out_file="resultados/w_geracao.json",
    topic_file="resultados/w_topicos.json",
)

print(
    "Quantidade de tópicos gerados:",
    qtd_topicos_wiki,
)

/home/tiagosampaio/experimento_mestrado/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/tiagosampaio/experimento_mestrado/.venv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:254: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/home/tiagosampaio/experimento_mestrado/.venv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:254: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.cloud.resou

 -> Iniciando Fase 1: Geração de Tópicos...
    [✔] Concluído em 1869.66s | Tópicos únicos encontrados: 324
Quantidade de tópicos gerados: 324


In [7]:
qtd_refinados_wiki = pipeline.refinar_topicos(
    prompt_file="prompts/refinement.txt",
    generation_file="resultados/w_geracao.json",
    topic_file="resultados/w_topicos.json",
    out_file="resultados/w_refinamento.json",
    updated_file="resultados/w_top_refinados.json",
    mapping_file="resultados/w_map.json",
)

print(
    "Quantidade de tópicos refinados:",
    qtd_refinados_wiki,
)

 -> Iniciando Fase 2: Refinamento de Tópicos...
    [✔] Concluído em 1078.81s | Tópicos: 324 -> 23 | Redução: 301 | Mapeamentos alterados: 0
    [✔] Árvore refinada original: resultados/w_refinamento.json
    [✔] Lista refinada para atribuição: resultados/w_top_refinados_limpo.txt
Quantidade de tópicos refinados: 23


In [8]:
from pathlib import Path

arquivo_topicos_refinados = Path(
    "resultados/w_top_refinados_limpo.txt"
)

if not arquivo_topicos_refinados.exists():
    raise FileNotFoundError(
        "O arquivo de tópicos refinados não foi criado."
    )

with open(
    arquivo_topicos_refinados,
    "r",
    encoding="utf-8",
) as arquivo:
    topicos_refinados = [
        linha.strip()
        for linha in arquivo
        if linha.strip()
    ]

print(
    "Quantidade de tópicos no arquivo refinado:",
    len(topicos_refinados),
)

print("\nPrimeiros cinco tópicos:")

for topico in topicos_refinados[:5]:
    print(topico)

Quantidade de tópicos no arquivo refinado: 23

Primeiros cinco tópicos:
[1] Science Fiction (Count: 0): Deals with imaginative concepts such as advanced technology, space travel, and futuristic societies.
[1] Television (Count: 0): Relates to the production and consumption of television programs.
[1] Naval History (Count: 0): Describes historical events and ships related to naval warfare.
[1] Video Games (Count: 0): Describes interactive electronic games played on various platforms.
[1] Music (Count: 0): Relates to musical compositions, performances, and instruments.


In [9]:
pipeline.atribuir_topicos(
    data_file=file_wiki_ass,
    prompt_file="prompts/assignment.txt",
    out_file="resultados/w_atribuida.json",
    topic_file="resultados/w_top_refinados_limpo.txt",
)

 -> Iniciando Fase 3: Atribuição de Tópicos...
    [✔] Concluído em 36225.02s | Tópicos disponíveis: 23 | Saída: resultados/w_atribuida.json


In [13]:
metricas_wiki = AvaliadorMetricas.avaliar_resultados(
    caminho_arquivo_json="resultados/w_atribuida.json",
    dataset_name="WIKI",
)

metricas_wiki


--- Resultados da Avaliação: WIKI ---
NMI: 0.6129 | ARI: 0.3788 | HMP: 0.5946


{'NMI': 0.6129385028138501,
 'ARI': 0.3788055321481017,
 'HMP': 0.5945777110794831,
 'Purity': 0.5552093718843469,
 'Inverse_Purity': 0.6399551345962113,
 'Amostras_Validadas': 8024,
 'Topicos_Criados': 134,
 'Topicos_Ground_Truth': 45}

In [17]:
import importlib
import inspect
import funcoes_rotina_final

importlib.invalidate_caches()
importlib.reload(funcoes_rotina_final)

from funcoes_rotina_final import (
    AvaliadorMetricas,
)

print(
    "Arquivo carregado:",
    funcoes_rotina_final.__file__,
)

print(
    "Assinatura carregada:",
    inspect.signature(
        AvaliadorMetricas.avaliar_resultados
    ),
)

Arquivo carregado: /home/tiagosampaio/experimento_mestrado/funcoes_rotina_final.py
Assinatura carregada: (caminho_arquivo_json, dataset_name, topic_file=None)


In [18]:
metricas_wiki = (
    AvaliadorMetricas
    .avaliar_resultados(
        caminho_arquivo_json=(
            "resultados/w_atribuida.json"
        ),
        dataset_name="WIKI",
        topic_file=(
            "resultados/"
            "w_top_refinados_limpo.txt"
        ),
    )
)

metricas_wiki


--- Resultados da Avaliação: WIKI ---
NMI: 0.5623 | ARI: 0.3310 | HMP: 0.5502
Tópicos disponíveis: 23 | Tópicos efetivamente utilizados: 23
Amostras validadas: 7829 | Respostas sem tópico válido: 195 | Respostas vazias: 0


{'NMI': 0.562258498315154,
 'ARI': 0.330985921885316,
 'HMP': 0.5502359790200504,
 'Purity': 0.46825903691403753,
 'Inverse_Purity': 0.6670072806233236,
 'Amostras_Validadas': 7829,
 'Topicos_Disponiveis': 23,
 'Topicos_Utilizados': 23,
 'Topicos_Ground_Truth': 45,
 'Respostas_Sem_Topico_Valido': 195,
 'Respostas_Sem_Conteudo': 0}

In [11]:
registrar_log(
    dataset="WIKI",
    metricas=metricas_wiki,
    n_gen=n_gen_w,
    n_ass=n_ass_w,
)

print("Resultado da Wiki registrado.")


[💾] Log salvo com sucesso em 'experimento_metricas_geral.json' (Fora da pasta resultados).
Resultado da Wiki registrado.


[
    {
        "dataset": "BILLS",
        "timestamp": "2026-09-18 05:42:01",
        "modo_teste": false,
        "amostra_geracao": 1000,
        "amostra_atribuicao": 15242,
        "parametros_llm": {
            "max_tokens": 300,
            "temperature": 0.0,
            "top_p": 0.0,
            "auto_correction_limit": 10,
            "min_doc_frequency_bills": 10,
            "min_doc_frequency_wiki": 5,
            "max_words_per_doc": 250
        },
        "metricas": {
            "NMI": 0.4840208828021054,
            "ARI": 0.3699149182536339,
            "HMP": 0.5454722895827044,
            "Purity": 0.6634299960635087,
            "Inverse_Purity": 0.4631281983991602,
            "Amostras_Validadas": 15242,
            "Topicos_Criados": 275,
            "Topicos_Ground_Truth": 21
        }
    },
    {
        "dataset": "WIKITEXT",
        "timestamp": "2026-09-18 17:42:33",
        "modo_teste": false,
        "amostra_geracao": 1100,
        "amostra_atribuicao": 8024,
        "parametros_llm": {
            "max_tokens": 300,
            "temperature": 0.0,
            "top_p": 0.0,
            "auto_correction_limit": 10,
            "min_doc_frequency_bills": 10,
            "min_doc_frequency_wiki": 5,
            "max_words_per_doc": 250
        },
        "metricas": {
            {'NMI': 0.562258498315154,
            'ARI': 0.330985921885316,
            'HMP': 0.5502359790200504,
            'Purity': 0.46825903691403753,
            'Inverse_Purity': 0.6670072806233236,
            'Amostras_Validadas': 7829,
            'Topicos_Disponiveis': 23,
            'Topicos_Utilizados': 23,
            'Topicos_Ground_Truth': 45
            }
        }
]